In [1]:
from datetime import datetime as dt
from typing import Tuple, Optional, Dict
from dfply import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.mixture import BayesianGaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from scipy.stats import norm
from scipy import stats
import pandas as pd
import numpy as np
import math, os, uuid, sys, requests, re, time, statistics
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

Open Tables

In [2]:
file_name =  "DATA_RESPONSES.xlsx"
# Responses
RESPONSES = pd.read_excel(file_name, sheet_name = "RESPONSES")
RESPONSES = pd.DataFrame(RESPONSES)

#Concepts
CONCEPTS = pd.read_excel(file_name, sheet_name = "CONCEPTS")
CONCEPTS = pd.DataFrame(CONCEPTS)

#Questions
QUESTIONS = pd.read_excel(file_name, sheet_name = "QUESTIONS")
QUESTIONS = pd.DataFrame(QUESTIONS)

# Surveys
SURVEYS = pd.read_excel(file_name, sheet_name = "SURVEYS")
SURVEYS = pd.DataFrame(SURVEYS)


Create Data Model

In [3]:

QUESTIONS = QUESTIONS >> left_join(CONCEPTS, by='CONCEPT_KEY') 
QUESTIONS = QUESTIONS >> left_join(SURVEYS, by='SURVEY_KEY') 
RESPONSES = RESPONSES >> left_join(QUESTIONS, by='QUESTION_KEY') 

Create Variables

In [4]:
RESPONSES['TIMESTAMPS'] = pd.to_datetime(RESPONSES['TIMESTAMPS'], format = '%Y-%m-%d %H:%M:%S')
RESPONSES['YEAR'] = RESPONSES['TIMESTAMPS'].dt.strftime("%Y")
RESPONSES['MONTH'] = RESPONSES['TIMESTAMPS'].dt.strftime('%m')
RESPONSES['HOUR'] = RESPONSES['TIMESTAMPS'].dt.strftime("%H")
RESPONSES['YEAR_MONTH'] = RESPONSES['YEAR'].astype(str) + "-" + RESPONSES['MONTH'].astype(str)
RESPONSES['HOUR_RANGES'] = pd.cut(RESPONSES['TIMESTAMPS'].dt.hour,bins=[0, 12, 17, 24],labels=["(08 - 12]", "(12 - 17]", "(17 - 21]"],include_lowest=True)


In [5]:
RESPONSES

,RESPONSE_KEY,QUESTION_KEY,RESPONSE_ID,SCHOOL_ID,TIMESTAMPS,RESPONSE,RESPONSE_ENCODED,SURVEY_KEY,CONCEPT_KEY,QUESTION_NUMBER,...,SURVEY_TYPE,SURVEY_NAME,SURVEY_VERSION,SURVEY_ID,SURVEY_PHASE,YEAR,MONTH,HOUR,YEAR_MONTH,HOUR_RANGES
0,R00001,Q023,1,5,2025-05-08 08:58:00,2,2,S003,C039,1,...,ACA,Academic Achievement,v1,ACA02,PRE,2025,05,08,2025-05,(08 - 12]
1,R00002,Q024,1,5,2025-05-08 08:58:00,2,2,S003,C038,2,...,ACA,Academic Achievement,v1,ACA02,PRE,2025,05,08,2025-05,(08 - 12]
2,R00003,Q025,1,5,2025-05-08 08:58:00,2,2,S003,C037,3,...,ACA,Academic Achievement,v1,ACA02,PRE,2025,05,08,2025-05,(08 - 12]
3,R00004,Q026,1,5,2025-05-08 08:58:00,3,3,S003,C022,4,...,ACA,Academic Achievement,v1,ACA02,PRE,2025,05,08,2025-05,(08 - 12]
4,R00005,Q027,1,5,2025-05-08 08:58:00,2,2,S003,C044,5,...,ACA,Academic Achievement,v1,ACA02,PRE,2025,05,08,2025-05,(08 - 12]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18226,R18227,Q229,1917,7,2025-04-28 19:01:00,NaN,NaN,S022,C049,8,...,MEN,Student Mental Health,v1,MEN01,POST,2025,04,19,2025-04,(17 - 21]
18227,R18228,Q230,1917,7,2025-04-28 19:01:00,Agree,4,S022,C051,9,...,MEN,Student Mental Health,v1,MEN01,POST,2025,04,19,2025-04,(17 - 21]
18228,R18229,Q231,1917,7,2025-04-28 19:01:00,4,4,S022,C068,10,...,MEN,Student Mental Health,v1,MEN01,POST,2025,04,19,2025-04,(17 - 21]
18229,R18230,Q232,1917,7,2025-04-28 19:01:00,the painting,the painting,S022,C067,11,...,MEN,Student Mental Health,v1,MEN01,POST,2025,04,19,2025-04,(17 - 21]
